# <span style="color: purple;">**ZooCell Segmentation Workshop 2026:**</span>
# A Step-by-Step Pipeline for 3D Segmentation & Transfer Learning

This notebook provides a (hopefully) comprehensive tutorial on processing volumetric electron microscopy (EM) data for cell segmentation using deep learning.

Specifically, this notebook will guide you through:

1. **Loading and Visualizing Volumetric EM Data**: Learn how to handle large 3D datasets from various formats.

2. **One-Shot Prediction with BioImage.IO Models**: Use pre-trained models from the BioImage.IO model zoo for boundary prediction.

3. **Fine-Tuning on Ground-Truth Data**: Adapt the model to your specific dataset using labeled training cubes.

4. **Robust Pipeline Implementation**: Best practices for reproducibility, error handling, and bioimage.io compliance.

## <span style="color: darkgreen;">Learning Objectives</span>

### By the end of this notebook, you will be able to:

- **Load and preprocess 3D EM volumes** from various file formats

- Utilize community-shared models from BioImage.IO

- **Implement fine-tuning workflows** for domain adaptation & task transfer

- Evaluate segmentation quality and export results

- **Package models** according to BioImage.IO standards

## <span style="color: darkgreen;">Table of Contents

- [Step 1: Loading Volumetric Data](#step-1-loading-volumetric-data)
- [Step 2: Loading Pre-trained Models from BioImage.IO](#2-loading-pre-trained-models-from-bioimageio)
- [Step 3: One-Shot Prediction with Sliding Window Inference](#3-one-shot-prediction-with-sliding-window-inference)
- [Step 4: Preparing Ground-Truth Data for Fine-Tuning](#4-preparing-ground-truth-data-for-fine-tuning)
- [Step 5: Fine-Tuning the Model](#5-fine-tuning-the-model)
- [Step 6: Evaluation and Comparison](#6-evaluation-and-comparison)
- [Step 7: Instance Segmentation with Watershed and Multicut](#7-instance-segmentation-with-watershed-and-multicut)
- [Step 8: Saving and Exporting Results](#8-saving-and-exporting-results)
- [Conclusion and Next Steps](#conclusion-and-next-steps)

## <span style="background: linear-gradient(to right, red, orange, yellow, green, blue, indigo, violet); -webkit-background-clip: text; -webkit-text-fill-color: transparent;">Let's get started!</span>

In [ ]:
# Environment Setup and Imports

# Standard scientific computing libraries
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

# Package for deep learning frameworks
import torch


In [ ]:

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("✅ GPU available!")

---
## Step 1: <span style="color: purple;">Loading Volumetric Data
We must load our 3D Electron Microscopy (EM) images and our manually annotated Ground Truth labels into memory.

**To do this, we will use the function `utils.load_em_data()`:** 

**Q:** What does `load_em_data` do?
 
**A:** It reads image files from your hard drive, converts them into a 3D NumPy array, and can (optionally) the pixel intensities (scaling them from 0 to 1) so the neural network can process the numbers effectively.

### Understanding Data Formats
- **OME-TIFF**: Standard format with metadata, supports multi-dimensional data
- **HDF5 (.h5/.hdf5)**: Hierarchical format, good for large datasets
- **N5**: Similar to HDF5, optimized for cloud storage
- **Zarr**: Cloud-native format for chunked arrays

In [ ]:
from volumetric_utils import load_volumetric_data, visualize_volume_slices

# Example of how to load and visualize a raw volume
print("Loading example EM volume...")
# Update this path to your actual data
data_path = "./data/clytia_data/organized_bbs/raw/ome_tiff/2026-02-27_12-47__FIBSEM_Run2_Feb2026_pt1__volume.ome.tif"
# Optional internal path for multi-layer files (HDF5/N5/Zarr)
data_internal_path = None  # e.g. 'raw' or 'volume' or '/entry/data'
series_idx = 0  # For OME-TIFF, specify which series to load if multiple are present

try:
    volume, metadata = load_volumetric_data(data_path, internal_path=data_internal_path, series_idx=series_idx, normalize=True, invert_contrast=True)
    print(f"✅ Loaded volume with shape: {volume.shape}")
    print(f"Data type: {metadata['dtype']}")
    print(f"Normalization range: {metadata['normalization_range']}")

    # Ensure volume is float32 for PyTorch compatibility
    if volume.dtype != np.float32:
        print(f"Converting volume from {volume.dtype} to float32...")
        volume = volume.astype(np.float32)

    # Visualize
    visualize_volume_slices(volume, "EM Volume Slices", num_slices=5, cmap='gray')

except Exception as e:
    print(f"⚠️ Could not load data: {e}")
    print("Please update the data_path variable with a valid file path.")

In [ ]:
# Example usage
print("Loading example EM volume ground-truth labels...")
# Update this path to your actual data
data_path = "./data/clytia_data/organized_bbs/labels/annotation_exs_bb1.n5"
# Optional internal path for multi-layer files (HDF5/N5/Zarr)
data_internal_path = '/setup0/timepoint0/s0'  # e.g. 'raw' or 'volume' or '/entry/data'
series_idx = 0  # For OME-TIFF, specify which series to load if multiple are present

try:
    labels, metadata = load_volumetric_data(data_path, internal_path=data_internal_path, series_idx=series_idx, normalize=False, load_labels=True)
    print(f"✅ Loaded volume with shape: {volume.shape}")
    print(f"Data type: {metadata['dtype']}")
    if 'normalization_range' in metadata:
        print(f"Normalization range: {metadata['normalization_range']}")

    # Visualize
    visualize_volume_slices(labels, "EM Volume Slices", num_slices=5, cmap='viridis', interpolation='nearest')

except Exception as e:
    print(f"⚠️ Could not load data: {e}")
    print("Please update the data_path variable with a valid file path.")

In [ ]:
from volumetric_utils import create_colored_labels

colored_labels, color_map, label_ids = create_colored_labels(labels)

# Visualize multiple slices
fig, axes = plt.subplots(1, 5, figsize=(15, 4))
fig.suptitle('Instance Labels Visualization (Each cell has unique color)', fontsize=14, fontweight='bold')

z_slices = np.linspace(0, labels.shape[0]-1, 5, dtype=int)
for ax, z in zip(axes, z_slices):
    ax.imshow(colored_labels[z])
    ax.set_title(f'Z slice {z}')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Each color represents a different cell instance.")


In [ ]:
from skimage.segmentation import find_boundaries
from volumetric_utils import smooth_binary_boundaries

# Extract binary boundaries from label map
boundaries = find_boundaries(labels, mode='thick').astype(np.float32)

visualize_volume_slices(boundaries, "Raw Label Boundaries", num_slices=5, cmap='gray', interpolation='nearest')

In [ ]:

# Apply smoothing and visualize
smoothed_boundaries = smooth_binary_boundaries(boundaries, gaussian_sigma=1.0, morph_radius=1.0, threshold=0.1)

visualize_volume_slices(smoothed_boundaries, "Smoothed EM Boundaries", num_slices=5, cmap='gray', interpolation='nearest')


In [ ]:
from volumetric_utils import visualize_boundary_conversion

# visualize the conversion of boundaries to a format suitable for training
visualize_boundary_conversion(volume, colored_labels, smoothed_boundaries)

---
## Step 2: <span style="color: purple;">Loading Pre-trained Models from BioImage.IO

BioImage.IO is a community platform for sharing trained deep learning models. We'll use a pre-trained model for membrane (boundary) prediction on our vEM data.

### Key Concepts:
- **Model Zoo**: Collection of models trained on diverse biological imaging data

- **RDF (Resource Description File)**: Standardized metadata describing the model

- **One-shot Prediction**: Using a model without training on your specific data

- **Model Compatibility**: Ensuring input/output shapes match your data

### Selected Model: <span style="color: darkgreen;">**CebraNet**
Cellular membrane prediction model for volume SEM datasets. This model was trained on a FIB-SEM dataset to generically predict membranes (or organelle boundaries) in any volume SEM dataset.

- The model was published as major component of the [CebraEM](https:/github.com/jhennies/CebraEM) workflow in [hennies et al. 2023](https://www.biorxiv.org/content/10.1101/2023.04.06.535829v1).

- CebraNET is available  in the Bioimage Model Zoo (bioimage.io)
([CebraNET @bioimage.io](https://bioimage.io/#/?id=10.5281%2Fzenodo.7274275), [CebraNET @zenodo](https://zenodo.org/record/7274276))

In [ ]:
# Model Loading from CebraEM
from volumetric_utils import load_cebraem_model

# Load the CebraEM model with actual weights
print("="*60)
print("LOADING CEBRAEM MODEL WITH WEIGHTS")
print("="*60)

model, model_metadata, device = load_cebraem_model(model_weights_path="./models/CebraNet/model_0097.pt",
                                            rdf_path="./models/CebraNet/rdf.yaml",
                                            model_architecture_path="./models/CebraNet/"
                       )    

print("\nCebraEM Model Information:")
for key, value in model_metadata.items():
    if key not in ['fallback']:
        print(f"  {key}: {value}")

In [ ]:
# Verify model works with a small test input
test_input = np.expand_dims(volume[:128, :128, :128], axis=(0, 1))  # Small test patch
test_input_tensor = torch.from_numpy(test_input).float().to(device)  # Ensure float32

print(f"Test input shape: {test_input.shape}")
print(f"Test input dtype: {test_input_tensor.dtype}")

with torch.no_grad():
    model.eval()
    try:
        test_output = model(test_input_tensor)
        test_output = test_output.cpu().numpy()

    except Exception as e:
        print(f"Error occurred during model forward pass: {str(e)}")
        raise

print(f"Test output shape: {test_output.shape}")
print(f"Test output range: [{test_output.min():.3f}, {test_output.max():.3f}]")
print("✅ Model forward pass successful!")

---
## Step 3: <span style="color: purple;">One-Shot Prediction with Sliding Window Inference

Since EM volumes are too large to fit in GPU memory, we use **sliding window inference**:
- Divide the volume into overlapping 3D patches
- Run prediction on each patch
- Stitch results back together with blending

### Key Parameters:
- **patch_size**: Size of 3D patches (must match model input)
- **overlap**: Overlap between patches for seamless stitching
- **batch_size**: Number of patches processed simultaneously


In [ ]:
from volumetric_utils import sliding_window_inference

# Run one-shot prediction
print("Running one-shot prediction on EM volume...")
try:
    boundary_prediction = sliding_window_inference(
        model=model,
        volume=volume,
        patch_size=(256, 256, 256),  # Minimum size required by CebraEM model
        overlap=(128, 128, 128), # Higher overlap reduce edge artifacts
        batch_size=1,  # Reduce batch size for large patches
        device='cuda',
        verbose=True
    )

    print(f"Prediction shape: {boundary_prediction.shape}")
    print(f"Prediction range: [{boundary_prediction.min():.3f}, {boundary_prediction.max():.3f}]")

except Exception as e:
    print(f"⚠️ Error during sliding window inference: {e}")
    boundary_prediction = None


In [ ]:

# Visualize results
if boundary_prediction is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    slice_idx = volume.shape[0] // 2
    axes[0].imshow(volume[slice_idx], cmap='gray')
    axes[0].set_title("Raw EM Image")

    axes[1].imshow(boundary_prediction[slice_idx], cmap='magma', vmin=0, vmax=1)
    axes[1].set_title("Boundary Prediction")

    # Overlay
    axes[2].imshow(volume[slice_idx], cmap='gray')
    axes[2].imshow(boundary_prediction[slice_idx], cmap='magma', alpha=0.5, vmin=0, vmax=1)
    axes[2].set_title("Overlay")

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Predictions not available. Check the cell above for errors.")
    

---
## Step 4: <span style="color: purple;">Preparing Ground-Truth Data for Fine-Tuning

To improve predictions, we need train on our labeled training data. This typically consists of:
- **Input patches**: Small, denseley annotated, 3D cubes from your EM volume
- **Target masks**: Binary masks indicating cell boundaries (as we looked at above)

### Data Preparation Steps:
1. Load ground-truth boundary labels (done already)
2. Extract corresponding patches from raw data (done already)
3. Create a PyTorch Dataset for efficient loading
4. Implement data augmentation for robustness

### Ground-Truth Formats:
- Instance masks (unique ID per cell: e.g. - our output cubes from WebKnossos)
- Binary masks (0 = background/cell interior, 1 = boundary : what we extracted from the cobes above ^^^)


In [ ]:
from volumetric_utils import get_training_transforms, PatchDataset
from torch.utils.data import DataLoader

try:
    if labels is not None and labels.shape == volume.shape:
        print(f"✅ Loaded ground-truth labels with shape: {labels.shape}")

        # Create dataset with augmentation (rotation, flipping, noise) to the training dataset
        train_transforms = get_training_transforms()

        train_dataset = PatchDataset(
            volume=volume,
            labels=smoothed_boundaries,  # Use smoothed boundaries as GT
            patch_size=(128, 128, 128),  # Match model input requirements
            samples_per_epoch=50,  # Number of random patches per epoch
            transform=train_transforms  # Use TorchVision-based 3D transforms
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=1, # batch size represents the number of patches in each training batch. You can increase this if you have enough GPU memory.
            shuffle=True,
            num_workers=5  # Set to 0 for debugging
        )

        print("✅ Training dataset created!")

        # Visualize a set of training samples/patches
        sample_vol, sample_label = train_dataset[0]
        fig, axes = plt.subplots(3, 2, figsize=(6, 10))
        plt.tight_layout() 

        num_slices = 5
        slice_indices = np.linspace(0, sample_vol.shape[1]-1, num_slices, dtype=int)
        for i, slice_idx in enumerate(slice_indices):
            axes[i, 0].imshow(sample_vol[0, :, :, slice_idx], cmap='gray')
            #axes[i, 0].axis('off')
            if i == 0:
                axes[i, 0].set_title("Training Patch")
            axes[i, 1].imshow(sample_label[0, :, :, slice_idx], cmap='viridis')
            #axes[i, 1].axis('off')
            if i == 0:
                axes[i, 1].set_title("Ground-Truth Boundaries")       
        
        plt.show()

    else:
        print("⚠️ Ground-truth labels not available or shape mismatch")
except Exception as e:
    print(f"⚠️ Error preparing ground-truth data: {e}")
    print("Please ensure that the labels are correctly loaded and match the volume shape.")

---
## Step 5: <span style="color: purple;">Fine-Tuning the Model

Fine-tuning adapts the pre-trained model to your specific dataset (domain) and or new segmentation target (task). 

## <span style="color: darkgreen;"> Key considerations

### Training Objective:
- **Domain Adaptation**: Am I retraining a model to perform the same task on a new (type of) dataset? --> Each dataset could be seen as a new domain
- **Task Transfer**: Am I retraining a model to perform a new segmentation task on the same (type of) dataset? --> Each new segmentation target could be seen as a new task. 

### Training Strategy:
- **High quality, densely annotated cubes**: Start with 2-3 well distributed across the target dataset
- **Learning rate scheduling**: Start with small LR, gradually increase ***
- **Early stopping**: Prevent overfitting ***

### Monitoring Training:
- **Training loss**: Should decrease steadily ***
- **Visual inspection**: Compare predictions before/after fine-tuning ***
- **Validation metrics**: F1-score, IoU, precision/recall


In [ ]:
from volumetric_utils import fine_tune_model

# Fine-tune the model
print("Fine-tuning the model on your data...")
try:
    fine_tuned_model, history = fine_tune_model(
        model=model,
        train_loader=train_loader,
        num_epochs=25,  # Adjust based on your data size
        learning_rate=1e-4, # Learning rate represent how much the model weights are updated during training. A smaller learning rate can lead to more stable training but may require more epochs.
        device='cuda' if torch.cuda.is_available() else 'cpu',
        freeze_encoder=True  # Keep pre-trained features
    )

except Exception as e:
    print(f"⚠️ Fine-tuning failed: {e}")
    print("Using original model for demonstration...")
    fine_tuned_model = model

---
## Step 6: <span style="color: purple;">Evaluation and Comparison

Compare the performance of the original vs fine-tuned model:

### Qualitative Assessment:
- **Visual inspection**: Side-by-side comparison of predictions
- **Boundary quality**: Are boundaries sharp and continuous?
- **False positives/negatives**: Over/under-segmentation analysis --> Is the model predicting more or less boundaries than actually exist?

In [ ]:
# Run full-volume prediction with fine-tuned model
print("\nRunning full-volume prediction with fine-tuned model...")
finetuned_prediction = sliding_window_inference(
    model=fine_tuned_model,
    volume=volume,
    patch_size=(256, 256, 256),  # Match model input requirements
    overlap=(128, 128, 128),
    batch_size=1,
    device=device,
    verbose=True
)

print("✅ Evaluation complete!")

In [ ]:
from volumetric_utils import visualize_model_comparison

# Visualize comparison between original and fine-tuned model predictions
visualize_model_comparison(
    volume=volume,
    original_prediction=boundary_prediction,
    finetuned_prediction=finetuned_prediction,
    slice_idx=volume.shape[0] // 2
)

In [ ]:
# Example of how to load and visualize a raw volume
print("Loading example EM volume...")
# Update this path to your actual data
data_path = "./data/clytia_data/organized_bbs/raw/ome_tiff/2026-04-07_12-06__FIBSEM_Run2_Feb2026_pt1__volume.ome.tif"
# Optional internal path for multi-layer files (HDF5/N5/Zarr)
data_internal_path = ''  # e.g. 'raw' or 'volume' or '/entry/data'
series_idx = 0  # For OME-TIFF, specify which series to load if multiple are present

try:
    new_test_volume, test_volume_metadata = load_volumetric_data(data_path, internal_path=data_internal_path, series_idx=series_idx, normalize=True, invert_contrast=True)
    print(f"✅ Loaded volume with shape: {volume.shape}")
    print(f"Data type: {test_volume_metadata['dtype']}")
    print(f"Normalization range: {test_volume_metadata['normalization_range']}")

    # Ensure volume is float32 for PyTorch compatibility
    if new_test_volume.dtype != np.float32:
        print(f"Converting volume from {new_test_volume.dtype} to float32...")
        new_test_volume = new_test_volume.astype(np.float32)

    # Visualize
    visualize_volume_slices(new_test_volume, "EM Volume Slices", num_slices=5, cmap='gray')

except Exception as e:
    print(f"⚠️ Could not load data: {e}")
    print("Please update the data_path variable with a valid file path.")

In [ ]:
old_cebraem_pred_on_new_volume = sliding_window_inference(
    model=model,
    volume=new_test_volume,
    patch_size=(256, 256, 256),  # Minimum size required by CebraEM model
    overlap=(128, 128, 128), # Higher overlap reduce edge artifacts
    batch_size=1,  # Reduce batch size for large patches
    device=device,
    verbose=True
)

In [ ]:
new_prediction = sliding_window_inference(
    model=fine_tuned_model,
    volume=new_test_volume,
    patch_size=(256, 256, 256),  # Match model input requirements
    overlap=(128, 128, 128),
    batch_size=1,
    device=device,
    verbose=True
)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 12))

slice_idx = random.choice(range(new_test_volume.shape[0]))  

# Row 1: Original model
axes[0].imshow(new_test_volume[slice_idx], cmap='gray')
axes[0].set_title("Raw EM Image")

axes[1].imshow(new_prediction[slice_idx], cmap='viridis')
axes[1].set_title("Model Prediction")

axes[2].imshow(new_test_volume[slice_idx], cmap='gray')
axes[2].imshow(new_prediction[slice_idx], cmap='viridis', alpha=0.5)
axes[2].set_title("Model Overlay")

In [ ]:
visualize_model_comparison(
    volume=new_test_volume,
    original_prediction=old_cebraem_pred_on_new_volume,
    finetuned_prediction=new_prediction,
    slice_idx=slice_idx
)

---
## Step 7: <span style="color: purple;">Instance Segmentation with Watershed and Multicut

The boundary predictions give us fuzzy cell outlines. To get individual cells, we use:

### Watershed Algorithm:
- Treats boundaries as dams
- Drops seeds in cell centers
- Grows regions until hitting boundaries

### Multicut Algorithm:
- Breaks image into small pieces (supervoxels)
- Uses graph optimization to decide which pieces belong together
- Better for complex cell shapes

### Parameters to Tune:
- **threshold**: Boundary confidence threshold
- **sigma_seeds**: Smoothing for seed placement
- **beta**: Multicut glue parameter

In [ ]:
from volumetric_utils import apply_watershed_segmentation, apply_multicut_segmentation, visualize_segmentation_comparison

# Apply segmentation algorithms
print("Converting boundary predictions to instance segmentation...")

# Watershed segmentation
watershed_seg = apply_watershed_segmentation(finetuned_prediction)

# Multicut segmentation
multicut_seg = apply_multicut_segmentation(finetuned_prediction, beta=0.5, threshold=0.5)

# Visualize results
visualize_segmentation_comparison(
    volume=volume,
    prediction=finetuned_prediction,
    watershed_seg=watershed_seg,
    multicut_seg=multicut_seg,
    slice_idx=volume.shape[0] // 2
)

print("✅ Instance segmentation complete!")

In [ ]:
# Apply segmentation algorithms
print("Converting boundary predictions to instance segmentation...")

# Watershed segmentation
old_watershed_seg = apply_watershed_segmentation(boundary_prediction)

# Multicut segmentation
old_multicut_seg = apply_multicut_segmentation(boundary_prediction, beta=0.5, threshold=0.5)

# Visualize results
visualize_segmentation_comparison(
    volume=volume,
    prediction=boundary_prediction,
    watershed_seg=old_watershed_seg,
    multicut_seg=old_multicut_seg,
    slice_idx=volume.shape[0] // 2
)

print("✅ Instance segmentation complete!")

In [ ]:
# Apply segmentation algorithms
print("Converting boundary predictions to instance segmentation...")

# Watershed segmentation
new_watershed_seg = apply_watershed_segmentation(new_prediction)

# Multicut segmentation
new_multicut_seg = apply_multicut_segmentation(new_prediction, beta=0.5)

# Visualize results
visualize_segmentation_comparison(
    volume=new_test_volume,
    prediction=new_prediction,
    watershed_seg=new_watershed_seg,
    multicut_seg=new_multicut_seg,
    slice_idx=new_test_volume.shape[0] // 2
)

In [ ]:
# Apply segmentation algorithms
print("Converting boundary predictions to instance segmentation...")

# Watershed segmentation
old_cebra_on_new_volume_watershed_seg = apply_watershed_segmentation(old_cebraem_pred_on_new_volume)

# Multicut segmentation
old_cebra_on_new_volume_multicut_seg = apply_multicut_segmentation(old_cebraem_pred_on_new_volume, beta=0.5)

# Visualize results
visualize_segmentation_comparison(
    volume=new_test_volume,
    prediction=old_cebraem_pred_on_new_volume,
    watershed_seg=old_cebra_on_new_volume_watershed_seg,
    multicut_seg=old_cebra_on_new_volume_multicut_seg,
    slice_idx=new_test_volume.shape[0] // 2
)

---
## Step 8: <span style="color: purple;">Saving and Exporting Results

### BioImage.IO Model Export:
- **RDF Format**: Standardized model description
- **Weights**: Model parameters in various formats
- **Documentation**: Usage instructions and citations
- **Validation**: Automated testing of model functionality

### Export Formats:
- **PyTorch**: Native format for further use
- **BioImage.IO ZIP**: Complete model package

### Segmentation Results:
- We export the results as **HDF5/N5**: Efficient storage for large volumes and allows us to store them under multiple channels


In [ ]:
from volumetric_utils import save_segmentation_results
import os

# Save segmentation results
output_dir = "./data/segmentations/finetuned_cebraNet_segmentations"
os.makedirs(output_dir, exist_ok=True)

results_metadata = {
    'model_name': model_metadata.get('name', 'unknown'),
    'input_volume_shape': volume.shape,
    'processing_date': '2026-04-02',  # Current date
    'watershed_cells': len(np.unique(watershed_seg)) - 1,
    'multicut_cells': len(np.unique(multicut_seg)) - 1
}

save_segmentation_results(
    segmentation=multicut_seg,
    watershed_seg=watershed_seg,
    boundary_pred=finetuned_prediction,
    output_path=os.path.join(output_dir, "training_cube_1_segmentation_results_v2.h5"),
    metadata=results_metadata
)

In [ ]:
results_metadata = {
    'model_name': model_metadata.get('name', 'unknown'),
    'input_volume_shape': volume.shape,
    'processing_date': '2026-04-02',  # Current date
    'watershed_cells': len(np.unique(new_watershed_seg)) - 1,
    'multicut_cells': len(np.unique(new_multicut_seg)) - 1
}

save_segmentation_results(
    segmentation=new_multicut_seg,
    watershed_seg=new_watershed_seg,
    boundary_pred=new_prediction,
    output_path=os.path.join(output_dir, "testing_cube_2_segmentation_results_v2.h5"),
    metadata=results_metadata
)

In [ ]:
from volumetric_utils import save_model_for_bioimageio
# Export the fine-tuned model
model_export_dir = "./models/finetuned_CebraNet"
os.makedirs(model_export_dir, exist_ok=True)

save_model_for_bioimageio(
    model=fine_tuned_model,
    model_name="finetuned_CebraNet",
    output_dir=model_export_dir,
    author_name="Seth Frazer",
    author_email="seth.frazer@embl.de",
    input_shape=(1, 1, 128, 128, 128),
    model_version="1.1.0",
    architecture_file="./models/cebraEM_finetuned/piled_unets.py",
    architecture_callable="PiledUnet"
)

---
# <span style="color: purple;">**Conclusion and Next Steps**

Congratulations! You have successfully implemented a complete end-to-end pipeline for 3D cell segmentation in electron microscopy data.

## What You Accomplished:

1. ✅ **Data Loading & Visualization**: Loaded and explored volumetric EM data from various formats
2. ✅ **Model Loading**: Utilized pre-trained models from BioImage.IO model zoo
3. ✅ **One-Shot Prediction**: Applied sliding window inference for large volume processing
4. ✅ **Fine-Tuning**: Adapted the model to your specific dataset with custom training
5. ✅ **Evaluation**: Qualitatively compared model performance before and after fine-tuning
6. ✅ **Instance Segmentation**: Converted boundary predictions to individual cell instances
7. ✅ **Export**: Saved results and models in standardized formats

## Key Takeaways:

- **BioImage.IO** provides a rich ecosystem of pre-trained models
- **Fine-tuning** significantly improves performance on specific datasets
- **Sliding window inference** enables processing of arbitrarily large volumes
- **Graph-based methods** Watershed and Multicut takes us from probablitiy maps, to superpixels, to our "final" segmentation masks.
- **Standardized export** ensures reproducibility and sharing

## Next Steps & Extensions:

### Immediate Improvements:
- **Cross-validation**: Evaluate on held-out test data
- **Hyperparameter optimization**: Tune learning rates, patch sizes, segmentation parameters
- **Data augmentation**: Add rotations, flips, intensity variations during training

### Advanced Topics:
- **Ensemble methods**: Combine predictions from multiple models
- **Interactive refinement**: Human-in-the-loop correction of segmentation errors
- **Batch processing**: Automate pipeline for multiple datasets
- **Quality control**: Implement automated checks for segmentation quality
- **Integration**: Connect with analysis pipelines (morphometrics, statistics)

## Resources:

- **BioImage.IO**: https://bioimage.io/ - Model zoo and specifications
- **ELF**: https://github.com/constantinpape/elf - Segmentation algorithms
- **PyTorch**: https://pytorch.org/ - Deep learning framework
- **BioIO**: https://github.com/bioio-devs/bioio - Modern bioimage I/O

Remember: The field of bioimage analysis is rapidly evolving. Stay updated with the latest models and techniques from the BioImage.IO community!

---

*This notebook was created for educational purposes. For production use, consider additional validation, error handling, and performance optimization.*